In [106]:
# xarray to read NETCDF
import numpy as np
import pandas as pd
import xarray as xr
import matplotlib.pyplot as plt
import seaborn as sns
import dask as dd
import cartopy.crs as ccrs
import cartopy.feature as cfeature
plt.rcParams['figure.dpi'] = 300

In [107]:
sst = xr.open_dataset('data/SST/sst.mnmean.nc')
sst = sst.assign_coords(lon=(((sst.lon + 180) % 360) - 180)).sortby(['lon'])
sst_df = sst.to_dataframe().reset_index()
sst_df = sst_df[sst_df['nbnds'] == 0]
start_year = 1993
end_year = 2024
sst_df['month'] = sst_df['time'].dt.month
sst_df['year'] = sst_df['time'].dt.year
mask = (sst_df['year'] >= start_year) & (sst_df['year'] <= end_year)
sst_df = sst_df[mask].reset_index().drop(['index'], axis=1)

In [109]:
chirps = xr.open_dataset('data/CHIRPS/chirps-v2.0.monthly.nc')

In [110]:
ltm_sst = sst_df.dropna().groupby(['lat', 'lon', 'month']).mean('sst').reset_index()
ltm_sst_clean = ltm_sst.drop(['time_bnds', 'nbnds', 'year'], axis=1)
ltm_sst_clean = ltm_sst_clean.rename(columns={'sst':'ltm_sst'})

In [111]:
sst_anomaly = sst_df.drop(['time_bnds', 'nbnds'], axis=1).merge(ltm_sst_clean, on=['lat', 'lon', 'month'], how='left')

In [112]:
sst_anomaly['sst_anomaly'] = sst_anomaly['sst'] - sst_anomaly['ltm_sst']

In [113]:
chirps_eastern_east_africa = chirps.sel(latitude=slice(-3.5, 8), longitude=slice(38, 50)).to_dataframe().reset_index()
chirps_eastern_east_africa['month'] = chirps_eastern_east_africa['time'].dt.month
chirps_eastern_east_africa['year'] = chirps_eastern_east_africa['time'].dt.year

In [114]:
months = ['jan', 'feb', 'mar', 'apr', 'may', 'jun', 'jul', 'aug', 'sep', 'oct', 'nov', 'dec']

for i in range(12):
    i += 1
    chirps_eastern_east_africa_month = chirps_eastern_east_africa.query(f'month == {i}').groupby(['month', 'year']).mean('precip').reset_index()
    # Create a boolean mask
    mask = (chirps_eastern_east_africa_month['year'] >= start_year) & (chirps_eastern_east_africa_month['year'] <= end_year)

    # Apply the mask to filter the DataFrame
    chirps_eastern_east_africa_month = chirps_eastern_east_africa_month[mask]

    # Get tercile values
    tercile_list = chirps_eastern_east_africa_month.quantile([0.33, 0.66])['precip'].to_list()

    month_bn = chirps_eastern_east_africa_month.query(f'precip <= {tercile_list[0]}')['year'].to_list()
    month_n = chirps_eastern_east_africa_month.query(f'{tercile_list[0]} <= precip <= {tercile_list[1]}')['year'].to_list()
    month_an = chirps_eastern_east_africa_month.query(f'{tercile_list[1]} <= precip')['year'].to_list()

    sst_anomaly_month = sst_anomaly.query(f'month == {i}')

    sst_df_month_bn = sst_anomaly_month[sst_anomaly_month['year'].isin(month_bn)]
    sst_df_month_n = sst_anomaly_month[sst_anomaly_month['year'].isin(month_n)]
    sst_df_month_an = sst_anomaly_month[sst_anomaly_month['year'].isin(month_an)]

    sst_df_month_dict = {'bn':sst_df_month_bn, 'n':sst_df_month_n, 'an':sst_df_month_an}

    for key, value in sst_df_month_dict.items():
        value.dropna().groupby(['lat', 'lon']).mean().drop(['time', 'sst', 'year', 'ltm_sst', 'month'], axis=1).reset_index().set_index(['lat', 'lon']).to_xarray()['sst_anomaly'].plot(
            subplot_kws=dict(projection=ccrs.PlateCarree()),
            transform=ccrs.PlateCarree()
        )
        plt.title(f'{key} sst anomaly {months[i-1]} eastern_east_africa')
        plt.savefig(f'figures/global_sst/sst_{months[i-1]}_{key}_eea.png')
        plt.close()

In [115]:
chirps_southern_africa = chirps.sel(latitude=slice(-23, -15), longitude=slice(25, 34)).to_dataframe().reset_index()
chirps_southern_africa['month'] = chirps_southern_africa['time'].dt.month
chirps_southern_africa['year'] = chirps_southern_africa['time'].dt.year

In [116]:
months = ['jan', 'feb', 'mar', 'apr', 'may', 'jun', 'jul', 'aug', 'sep', 'oct', 'nov', 'dec']

for i in range(12):
    i += 1
    chirps_southern_africa_month = chirps_southern_africa.query(f'month == {i}').groupby(['month', 'year']).mean('precip').reset_index()
    # Create a boolean mask
    mask = (chirps_southern_africa_month['year'] >= start_year) & (chirps_southern_africa_month['year'] <= end_year)

    # Apply the mask to filter the DataFrame
    chirps_southern_africa_month = chirps_southern_africa_month[mask]

    # Get tercile values
    tercile_list = chirps_southern_africa_month.quantile([0.33, 0.66])['precip'].to_list()

    month_bn = chirps_southern_africa_month.query(f'precip <= {tercile_list[0]}')['year'].to_list()
    month_n = chirps_southern_africa_month.query(f'{tercile_list[0]} <= precip <= {tercile_list[1]}')['year'].to_list()
    month_an = chirps_southern_africa_month.query(f'{tercile_list[1]} <= precip')['year'].to_list()

    sst_anomaly_month = sst_anomaly.query(f'month == {i}')

    sst_df_month_bn = sst_anomaly_month[sst_anomaly_month['year'].isin(month_bn)]
    sst_df_month_n = sst_anomaly_month[sst_anomaly_month['year'].isin(month_n)]
    sst_df_month_an = sst_anomaly_month[sst_anomaly_month['year'].isin(month_an)]

    sst_df_month_dict = {'bn':sst_df_month_bn, 'n':sst_df_month_n, 'an':sst_df_month_an}

    for key, value in sst_df_month_dict.items():
        value.dropna().groupby(['lat', 'lon']).mean().drop(['time', 'sst', 'year', 'ltm_sst', 'month'], axis=1).reset_index().set_index(['lat', 'lon']).to_xarray()['sst_anomaly'].plot(
            subplot_kws=dict(projection=ccrs.PlateCarree()),
            transform=ccrs.PlateCarree()
        )
        plt.title(f'{key} sst anomaly {months[i-1]} southern_africa')
        plt.savefig(f'figures/global_sst/sst_{months[i-1]}_{key}_sa.png')
        plt.close()